In [22]:
import os
import pandas as pd
from uuid import uuid4
import re
from collections import defaultdict
from typing import Tuple
from tqdm.asyncio import tqdm

In [2]:
from gecodb_compound_parser import perform_umlaut

In [21]:
sp18_path = "../resources/schaefer_pankratz_2018/Concordance.full.csv"
celex_gml_path = "../resources/Celex/german/gml/gml.cd"
celex_gmw_path = "../resources/Celex/german/gmw/gmw.cd"

outdir = "../resources/developed/sp18"

In [4]:
sp18 = pd.read_csv(
    sp18_path,
    sep="\t",
    header=0,
    index_col="Match",
    dtype=defaultdict(lambda: str, LE="int"),   # all str, LE int
    quoting=3   # = csv.QUOTE_NONE, separate by tabs only
)
sp18.head()

,DocURL,DocID,SentID,LC,RC,N2Num,N2Typ,N1Lemma,N1Pl,LE
Match,,,,,,,,,,
Apfelästen,http://www.rattenforum.de/phpbb/viewtopic.php?...,3fc371483b113c1f9bbda743f88603d7014f,11,"Eben , als ich neues Futter und frisches Wasse...",waren einige und ganz vereinzelte an den Käfig...,Pl,Ind,Apfel,U,0
Apfelbad,http://old.glamour.de/glamour/7/2/content/0170...,552f4c720b020f5922a890c09ff8d60c2562,3,Dann spielen Sie mit uns und gewinnen Sie zwei...,relaxen ! Ihr Gutschein ist gültig bis Mai 200...,Sg,Ind,Apfel,U,0
Apfelbauer,http://www.genau-dein-obst.de/apfelblog/page/34,2a56dd938e65d789a0a9cca3015f17acdb16,44,Bei schönstem Sonnenschein nimmt Hannes Panter...,. Auf den Obstplantagen wachsen zum größten Te...,Sg,Ind,Apfel,U,0
Apfelbaum,http://blog.solingen-internet.de/totterblotsch...,fa540ade2bbfec78e9a3cb1a54706cd6da5e,60,Zuweilen gibt es aber auch romantische Momente...,""" für eine Hofschaft mit kleiner Straße nicht ...",Sg,Ind,Apfel,U,0
Apfelbäumchen,http://blog.wiwo.de/chefsache/2009/03/14/der-s...,8cd3866b0dd929e9bc7a40904fddb8ae0460,72,Manche Politiker reden ihre Anliegen ohne Not ...,und seinem diskontierten Wert nennt sich übrig...,Sg,Ind,Apfel,U,0


In [5]:
def umlambda(x: str) -> str:
    x = x.lower()
    x = re.sub("(?<![aeiouy])ae", "ä", x, flags=re.IGNORECASE)
    x = re.sub("(?<![aeiouy])oe", "ö", x, flags=re.IGNORECASE)
    x = re.sub("(?<![aeiouy])ue", "ü", x, flags=re.IGNORECASE)
    return x

In [6]:
gml = pd.read_csv(
    celex_gml_path,
    sep="\\",
    header=None,
    dtype=str,
    usecols=[0, 1],
    names=["id", "lemma"],
    index_col="id"
).dropna()

# keep only nouns
gml = gml[gml["lemma"].str[0].str.isupper()]

gml

,lemma
id,
1,A
2,Ae
4,Aal
6,Aar
7,Aas
...,...
51724,Zytologie
51725,Zytoplasma
51726,Zytosom


In [7]:
gmw = pd.read_csv(
    celex_gmw_path,
    sep="\\",
    header=None,
    dtype=str,
    usecols=[1, 3, 4],
    names=["wordform", "id", "forms"],
    index_col="id"
).dropna()

gmw

,wordform,forms
id,,
35,abbestelle,"1SIE,13SKE"
35,abbestellst,2SIE
35,abbestellen,"13PIE,13PKE,i"
35,abbestellte,"13SIA,13SKA"
35,abbestelltet,"2PIA,2PKA"
...,...,...
27964,Papste,dS
27964,Paepsten,dP
27964,Papstes,gS


In [8]:
# joining will automatically drop non-noun wordforms,
# because they have no lemma in gml
gmwl = gmw.join(gml["lemma"], how="inner")  \
    .reset_index(drop=True)  \
    .set_index("wordform")  \
    .sort_index()

gmwl.index = gmwl.index.map(umlambda)
gmwl["lemma"] = gmwl["lemma"].map(umlambda)
gmwl = gmwl[["lemma", "forms"]]

gmwl

,lemma,forms
wordform,,
a,a,"nP,gP,dP,aP,nS,gS,dS,aS"
aal,aal,"nS,dS,aS"
aale,aal,"nP,gP,aP,dS"
aalen,aal,dP
aales,aal,gS
...,...,...
schöbest kegel,kegelschieben,2SKA/
schöbet kegel,kegelschieben,2PKA/
schöbst kegel,kegelschieben,2SKA/


In [9]:
# Comparing the original Match column values with
# with reconstructions from `parse_sp18_comp()`
# (before lemmatization to NomSg with the help of CELEX2!)
# during development of this module
# revealed 154 mismatches in compounds built from
# six different N1. Manual inspection of these mismatches revealed that
# some compounds with N1s 'Dämon', 'Gerät', 'Geräusch', 'Schloss'
# were spelled alternatively, and that compounds
# with N1s 'Auge' and 'Birne' productively exhibit
# alternation between _+n_ and _-e_ (!).
# See more in 'resources/developed/sp18/leftovers/mismatches.csv'.
# Thus, we will correct
# the spellings of the first four N1s and remove
# all compounds with the last two N1s.

# After these two changes, all reconstructed compounds
# after `parse_sp18_comp()` do match the original Match column
# (before lemmatization to NomSg with the help of CELEX2!).

# correct spellings
for old, new in [
    ("Daemon", "Dämon"),
    ("Geraet", "Gerät"),
    ("Geraeusch", "Geräusch"),
    ("Schloß", "Schloss")
]:
    sp18.index = sp18.index.map(
        lambda x: x.replace(old, new)
        if x.startswith(old) else x
    )

# remove compounds with N1s 'Auge' (N=185) and 'Birne' (N=200)
sp18 = sp18[~sp18["N1Lemma"].isin(["Auge", "Birne"])]

In [10]:
# drop NaN (2 rows only)
sp18 = sp18.dropna()

In [11]:
# extract document metadata from the dataset;
# most probably, we won't need it (also
# many URLs are now down, anyways),
# so by decomposing the dataframe, we can
# reduce processing effort; on the other hand,
# joining the metadata back to the main dataframe
# is easy, so we can always do it later if needed

# first, join the left and right contexts into a single column,
# drop the original columns, and assign a unique id to each context
sp18["context"] = sp18.LC + " " + sp18.index + " " + sp18.RC
sp18.drop(columns=["LC", "RC"], inplace=True)
sp18["context_id"] = [str(uuid4()) for _ in range(len(sp18))]
sp18.head()

,DocURL,DocID,SentID,N2Num,N2Typ,N1Lemma,N1Pl,LE,context,context_id
Match,,,,,,,,,,
Apfelästen,http://www.rattenforum.de/phpbb/viewtopic.php?...,3fc371483b113c1f9bbda743f88603d7014f,11,Pl,Ind,Apfel,U,0,"Eben , als ich neues Futter und frisches Wasse...",ea2fb4b6-64f6-411e-beb7-3389f12627ec
Apfelbad,http://old.glamour.de/glamour/7/2/content/0170...,552f4c720b020f5922a890c09ff8d60c2562,3,Sg,Ind,Apfel,U,0,Dann spielen Sie mit uns und gewinnen Sie zwei...,1c9014c3-81a3-4f3d-9c88-4910aa52ec37
Apfelbauer,http://www.genau-dein-obst.de/apfelblog/page/34,2a56dd938e65d789a0a9cca3015f17acdb16,44,Sg,Ind,Apfel,U,0,Bei schönstem Sonnenschein nimmt Hannes Panter...,6512d9d2-0346-4da1-801a-3c3bab86741a
Apfelbaum,http://blog.solingen-internet.de/totterblotsch...,fa540ade2bbfec78e9a3cb1a54706cd6da5e,60,Sg,Ind,Apfel,U,0,Zuweilen gibt es aber auch romantische Momente...,52b2de01-9c7a-4a9a-8945-03fe3cd9f893
Apfelbäumchen,http://blog.wiwo.de/chefsache/2009/03/14/der-s...,8cd3866b0dd929e9bc7a40904fddb8ae0460,72,Sg,Ind,Apfel,U,0,Manche Politiker reden ihre Anliegen ohne Not ...,d91dafca-cf4f-48f7-b28d-ae276d5f7413


In [12]:
# now, extract the document metadata
# into a separate dataframe
sp18_meta = sp18[
        ["context_id", "DocID", "DocURL", "SentID"]
    ]   \
    .rename(
        columns={"DocID": "doc_id", "DocURL": "url", "SentID": "sent_n"}
    )   \
    .reset_index(drop=True) \
    .set_index("context_id")    \
    .sort_index()

sp18_meta.head()

,doc_id,url,sent_n
context_id,,,
0007a5c8-5895-4067-a53e-6b30b11ed3d5,791ca44bed7959111504696a832eccb70439,http://www.ach-satire.de/a_26/Satire_03_02_24....,35
000aa4c3-37e5-494f-b437-55178608dad2,9ef74ec98e0cd3be6c5c6472b708a1cd46a7,http://www.uni-protokolle.de/foren/viewt/56740...,135
000ddf2e-7d02-40e1-94df-a848c6fd64aa,11d622d27cd30eeacb56c30691dfa46d4198,http://www.super8.homepage.t-online.de/Tipps%2...,42
000e21bf-36cb-4eaf-ba59-3d696586fbbd,a07cbf3a36a0e396ce22216758a26f8de71a,http://www.trier.de/kulturbuero/Kulturraum_Gro...,1
0021f977-814a-4b11-adc9-4a30ad02bc7d,9385dec6a330fd2397b7aa6039bbbeab8957,http://www.umgehungsstrasse.de/071105.html,23


In [13]:
# finally, drop the metadata columns
# from the main dataframe
sp18.drop(columns=["DocID", "DocURL", "SentID"], inplace=True)
sp18.head()

,N2Num,N2Typ,N1Lemma,N1Pl,LE,context,context_id
Match,,,,,,,
Apfelästen,Pl,Ind,Apfel,U,0,"Eben , als ich neues Futter und frisches Wasse...",ea2fb4b6-64f6-411e-beb7-3389f12627ec
Apfelbad,Sg,Ind,Apfel,U,0,Dann spielen Sie mit uns und gewinnen Sie zwei...,1c9014c3-81a3-4f3d-9c88-4910aa52ec37
Apfelbauer,Sg,Ind,Apfel,U,0,Bei schönstem Sonnenschein nimmt Hannes Panter...,6512d9d2-0346-4da1-801a-3c3bab86741a
Apfelbaum,Sg,Ind,Apfel,U,0,Zuweilen gibt es aber auch romantische Momente...,52b2de01-9c7a-4a9a-8945-03fe3cd9f893
Apfelbäumchen,Sg,Ind,Apfel,U,0,Manche Politiker reden ihre Anliegen ohne Not ...,d91dafca-cf4f-48f7-b28d-ae276d5f7413


In [14]:
# First, we need to be able to reconstruct the whole
# compound composition. We need that for three reasons:
# 1. To be able to revert the compounds to NomSg,
# which, in its turn, is required to identify duplicates;
# 2. Beside duplicates removal, we will also be able to
# identify contrastive pairs from the same constituents
# but with pluralic vs. non-pluralic linkers. This would
# be needed to assess the external plural effect
# following the method suggested in [pankratz_2017:20]
# (even though the examples are given for the internal plural effect).
# There is a change that the findings with this approach
# will be different as in [schaefer_pankratz_2018:26] since they
# do not contrast such pairs but rather compare all compounds in Pl forms
# to those in Sg forms.
# 3. Finally, similarly to [shmalts_2026], we would want
# to filter out the constituents not present in CELEX-nouns.

def lemmatize_n2(n2: str, expected_num: str) -> str:
    if n2 not in gmwl.index:
        return n2, False
    n2s = gmwl.loc[n2]
    if isinstance(n2s, pd.Series):  # one row found
        return n2s["lemma"], True
    else:
        if n2s.lemma.nunique() == 1:
            # one variant, ok
            return n2s.lemma.iloc[0], True
        else:
            # try finding a variant
            # that would correspond
            # to the annotated number of N2
            ncode = "P" if expected_num == "pl" else "S"
            matching_n2s = n2s[n2s["forms"].str.contains(ncode)]
            if matching_n2s.lemma.nunique() == 1:
                # one variant, ok
                return matching_n2s.lemma.iloc[0], True
    # otherwise: multiple variants matching the expected number,
    # unreliable parsing
    return n2, False


def parse_sp18_comp(
    comp: str,
    n1: str,
    pl_marker: str,
    has_linker: bool,
    n2_num: str,
    pbar: tqdm
) -> Tuple[str, str, str, bool]:

    pl_gc_template = "_+={pl_marker}_" if "U" in pl_marker else "_+{pl_marker}_"
    maybe_uml = perform_umlaut if ("U" in pl_marker and has_linker) else lambda x: x
    pl_marker = pl_marker.replace("U", "")
    # Schäfer & Pankratz 2018 only consider two
    # contrastive options: zero linker vs. pluralic linker;
    # are other non-pluralic linkers present in the dataset?
    linker = pl_marker if has_linker else ""
    n1_cstem = maybe_uml(n1) + linker
    n2 = re.sub(f"^{n1_cstem}", "", comp)

    # now, take the resulting n2 and lemmatize it to NomSg with the help of CELEX2
    n2, is_lemmatized = lemmatize_n2(n2, expected_num=n2_num)

    # gecodb style
    pl_gc = pl_gc_template.format(pl_marker=pl_marker)
    gecodb = f"{n1}{pl_gc if has_linker else '_'}{n2}"

    pbar.update(1)

    return gecodb, pl_gc, n2, is_lemmatized

In [15]:
sp18.index = sp18.index.map(str.lower)
for col in ["N1Lemma", "N2Num", "N2Typ"]:
    sp18[col] = sp18[col].map(str.lower)
sp18.head()

,N2Num,N2Typ,N1Lemma,N1Pl,LE,context,context_id
Match,,,,,,,
apfelästen,pl,ind,apfel,U,0,"Eben , als ich neues Futter und frisches Wasse...",ea2fb4b6-64f6-411e-beb7-3389f12627ec
apfelbad,sg,ind,apfel,U,0,Dann spielen Sie mit uns und gewinnen Sie zwei...,1c9014c3-81a3-4f3d-9c88-4910aa52ec37
apfelbauer,sg,ind,apfel,U,0,Bei schönstem Sonnenschein nimmt Hannes Panter...,6512d9d2-0346-4da1-801a-3c3bab86741a
apfelbaum,sg,ind,apfel,U,0,Zuweilen gibt es aber auch romantische Momente...,52b2de01-9c7a-4a9a-8945-03fe3cd9f893
apfelbäumchen,sg,ind,apfel,U,0,Manche Politiker reden ihre Anliegen ohne Not ...,d91dafca-cf4f-48f7-b28d-ae276d5f7413


In [16]:
with tqdm(total=len(sp18), desc="Lemmatizing compounds") as pbar:
    sp18["gecodb"], sp18["n1_pl_marker"], sp18["n2"], sp18["is_lemmatized"] = zip(
        *sp18.apply(
            lambda row: parse_sp18_comp(
                comp=row.name,
                n1=row["N1Lemma"],
                pl_marker=row["N1Pl"],
                has_linker=int(row["LE"]),
                n2_num=row["N2Num"],
                pbar=pbar
            ),
            axis=1
        )
    )

sp18.head()

Lemmatizing compounds: 100%|██████████| 9028/9028 [00:13<00:00, 689.66it/s]


,N2Num,N2Typ,N1Lemma,N1Pl,LE,context,context_id,gecodb,n1_pl_marker,n2,is_lemmatized
Match,,,,,,,,,,,
apfelästen,pl,ind,apfel,U,0,"Eben , als ich neues Futter und frisches Wasse...",ea2fb4b6-64f6-411e-beb7-3389f12627ec,apfel_ast,_+=_,ast,True
apfelbad,sg,ind,apfel,U,0,Dann spielen Sie mit uns und gewinnen Sie zwei...,1c9014c3-81a3-4f3d-9c88-4910aa52ec37,apfel_bad,_+=_,bad,True
apfelbauer,sg,ind,apfel,U,0,Bei schönstem Sonnenschein nimmt Hannes Panter...,6512d9d2-0346-4da1-801a-3c3bab86741a,apfel_bauer,_+=_,bauer,True
apfelbaum,sg,ind,apfel,U,0,Zuweilen gibt es aber auch romantische Momente...,52b2de01-9c7a-4a9a-8945-03fe3cd9f893,apfel_baum,_+=_,baum,True
apfelbäumchen,sg,ind,apfel,U,0,Manche Politiker reden ihre Anliegen ohne Not ...,d91dafca-cf4f-48f7-b28d-ae276d5f7413,apfel_bäumchen,_+=_,bäumchen,True


In [17]:
sp18 = sp18[
        [
            "gecodb", "N1Lemma", "n2", "n1_pl_marker",
            "N2Typ", "N2Num", "LE", "is_lemmatized",
            "context", "context_id"
        ]
    ]   \
    .rename(
        columns={
            "N1Lemma": "n1", "N2Typ": "n1_sem",
            "N2Num": "n2_num", "LE": "is_linker_pl"
        }
    )   \
    .reset_index(drop=True) \
    .set_index("gecodb")    \
    .sort_index()

sp18.sample(5)

,n1,n2,n1_pl_marker,n1_sem,n2_num,is_linker_pl,is_lemmatized,context,context_id
gecodb,,,,,,,,,
ohr_+en_infektion,ohr,infektion,_+en_,ind,pl,1,True,Nach mehreren folgenden Laboruntersuchungen ko...,97f46c70-6cf6-44ae-bff8-45cf321a05cb
bad_+=er_abteilung,bad,abteilung,_+=er_,ind,sg,1,True,Wir bieten ein breitgefächertes Therapieprogra...,f44fce34-75b4-47f0-a4ef-7cc58ae0a92a
ohr_sessel,ohr,sessel,_+en_,ind,sg,0,True,Majestät vom Dampfer abzuholen . In den Nachen...,2e414bd0-860d-4811-b90a-1b7c8c8a3be0
element_kind,element,kind,_+e_,ind,pl,0,True,Diese sollen ihren eigenen Äthersöhnen und - t...,33de395c-a480-4e0a-8c4e-3f30a8cd35fa
wurm_+=er_hasser,wurm,hasser,_+=er_,ind,pl,1,True,Diese reagiert unerwartet heftig mit dramatisc...,0de665cf-0ec5-41a0-b0f3-cc53f6965bd5


In [26]:
# From here on, we will have two versions of the dataset
# because reliable deduplication and detection of contrastive Sg-Pl pairs
# depends on the lemmatization of N2:
# 1. A version with lemmatized compounds only. We will extract the
# contrastive pairs from this version and use
# it for the analysis of the external plural effect.
# The compounds remaining after this extraction
# will be deduplicated and used for the analysis of
# the internal plural effect.
# 2. A joint version with both lemmatized and not lemmatized compounds.
# Will be used for the analysis of the internal plural effect only.
# Caution should be exercised when interpreting the results
# of the internal plural effect analysis based on the joint version.

# not-lemmatized compounds (N=1266)
sp18_nl = sp18[sp18["is_lemmatized"] == False]
sp18_nl.drop(columns=["is_lemmatized"], inplace=True)
sp18_nl.to_csv(
    os.path.join(outdir, "leftovers/not_lemmatized.tsv"),
    sep="\t",
    index=True,
    header=True
)

# joint version (N=9028)
sp18_all = sp18.copy()

# default version: lemmatized compounds only (N=7762)
sp18 = sp18_all[sp18_all["is_lemmatized"] == True]
sp18.drop(columns=["is_lemmatized"], inplace=True)

,n1,n2,n1_pl_marker,n1_sem,n2_num,is_linker_pl,context,context_id
gecodb,,,,,,,,
apfel_+=_achtel,apfel,achtel,_+=_,ind,pl,1,"Eiweiß dazugeben , Mandeln und Speißestärke un...",94b7460b-5b67-48aa-8f4d-b9a1e502ccce
apfel_+=_allergie,apfel,allergie,_+=_,ind,sg,1,"In der Symptomsprache , nach der die Kreative ...",541aa90a-4436-44f7-b79d-7bb14e6f2299
apfel_+=_annahme,apfel,annahme,_+=_,ind,sg,1,Die Kapsel mit vielen Dokumenten wird vom Klem...,a6eedfe5-cfb1-4e16-bba6-1877480992be
apfel_+=_anzahl,apfel,anzahl,_+=_,coll,sg,1,"Schaut gut aus , aber sind halt auch nur einfa...",49ba8793-5a2e-4051-a279-4a32113dccd4
apfel_+=_ausstellung,apfel,ausstellung,_+=_,konvc,sg,1,Wiederum für viele sehr interessant waren die ...,39e77e65-eb31-41bc-861f-2ba8fc56565c
...,...,...,...,...,...,...,...,...
zitat_zutraulichkeit,zitat,zutraulichkeit,_+e_,ind,sg,0,"Wenn also insinuiert wird , daß die vulgär-dra...",80655b7b-e993-40a6-834f-841ea5bb0a04
zitat_zweck,zitat,zweck,_+e_,ind,sg,0,Zitate aus noch unveröffentlichten Filmen sind...,37688edd-9151-480a-bace-0ea62304a6f2
zitat_zweck,zitat,zweck,_+e_,ind,pl,0,"Ein rechtmäßiger Zitatzweck liegt nur vor , we...",2358ae25-292c-4fb9-8c88-b7e38a76e5e8


In [54]:
# extract contrastive Sg-Pl pairs from the lemmatized version
sp18["const_comb"] = sp18["n1"] + "+" + sp18["n2"]
ident_const = sp18[sp18.duplicated(subset="const_comb", keep=False)]
ident_const

,n1,n2,n1_pl_marker,n1_sem,n2_num,is_linker_pl,context,context_id,const_comb
gecodb,,,,,,,,,
apfel_+=_bauer,apfel,bauer,_+=_,ind,pl,1,"Kauf dir ein Apfel , und zahle Applepatente gl...",e5ca29b1-67f4-40cf-992d-691fb469f7b3,apfel+bauer
apfel_+=_baum,apfel,baum,_+=_,ind,pl,1,An einem kleinem Flüsschen wandern wir bis sic...,db5333fa-5fc9-4125-929c-4cd808938ee2,apfel+baum
apfel_+=_baum,apfel,baum,_+=_,ind,pl,1,Die Verlegung auf diesen Termin aus Witterungs...,06107a2e-5b88-4be8-b72d-39d53c74afb8,apfel+baum
apfel_+=_baum,apfel,baum,_+=_,ind,pl,1,Wir fahren an großen Obstgärten vorüber . Die ...,2c1a4da5-46f4-4df1-81f8-695165823482,apfel+baum
apfel_+=_berg,apfel,berg,_+=_,coll,sg,1,Wie von der Tarantel gestochen sprang ihre Oma...,e9acac0a-8d5f-40b5-b3b1-66e6a5a8efb8,apfel+berg
...,...,...,...,...,...,...,...,...,...
zitat_verzeichnis,zitat,verzeichnis,_+e_,konvc,pl,0,Die bestehende Ordnung sieht für die Einstellu...,a4b9b7ed-ebf1-4a75-a20d-9e376a9fada8,zitat+verzeichnis
zitat_zitat,zitat,zitat,_+e_,ind,pl,0,Hier fließt also ein Engagement in der Kunst m...,b94b0860-f964-4b5e-bb55-a60438f58010,zitat+zitat
zitat_zweck,zitat,zweck,_+e_,ind,sg,0,Zitate aus noch unveröffentlichten Filmen sind...,37688edd-9151-480a-bace-0ea62304a6f2,zitat+zweck
